In [1]:
!pip3 install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 138.1 kB/s eta 0:00:00m eta 0:00:010:01:01


In [3]:
import nltk
from nltk.tokenize import word_tokenize
import  re
import emoji
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
corpus = 'Who ❤️ "word embeddings" in 2020? I do!!!'
data = re.sub(r'[,!?;-]+', '.', corpus)
data = nltk.word_tokenize(data)
print(data)

['Who', '❤️', '``', 'word', 'embeddings', "''", 'in', '2020', '.', 'I', 'do', '.']


In [5]:
corpus = 'Who ❤️ "word embeddings" in 2020? I do!!!'
data = re.sub(r'[,!?;-]+', '.', corpus)
data = nltk.word_tokenize(data)
data = [ch.lower() for ch in data
        if ch.isalpha()
        or ch == '.'
        or emoji.is_emoji(ch)
       ]
print(data)

['who', '❤️', 'word', 'embeddings', 'in', '.', 'i', 'do', '.']


In [6]:
def get_windows(words, C):
    i = C
    while i < len(words) - C:
        center_word = words[i]
        context_words = words[(i-C):i] + words[(i+1):(i + C + 1)]
        yield context_words, center_word
        i +=1 

In [7]:
for x, y in get_windows(
    ['i', 'am' 'happy', 'because', 'i', 'am', 'learning'],2):
    print(f'{x}\t{y}')

['i', 'amhappy', 'i', 'am']	because
['amhappy', 'because', 'am', 'learning']	i


In [8]:
corpus = 'Who ❤️ "word embeddings" in 2020? I do!!!'
data = re.sub(r'[,!?;-]+', '.', corpus)
data = nltk.word_tokenize(data)
print(data)

['Who', '❤️', '``', 'word', 'embeddings', "''", 'in', '2020', '.', 'I', 'do', '.']


In [9]:
def tokenize(corpus):
    data = re.sub(r'[,!?;-]+', '.', corpus)
    data = nltk.word_tokenize(data)
    data = [ch.lower() for ch in data
            if ch.isalpha()
            or ch == '.'
            or emoji.get_emoji_regexp().search(ch)
           ]
    return data

In [10]:
corpus = "I am happy because i am learning"
print(f'corpus:{corpus}')
words = tokenize(corpus)
print(f'Words (tokens): {words}')

corpus:I am happy because i am learning
Words (tokens): ['i', 'am', 'happy', 'because', 'i', 'am', 'learning']


In [11]:
def get_dict(data):
    words = sorted(list(set(data)))
    n = len(words)
    idx = 0
    word2Ind = {}
    Ind2word = {}
    for k in words:
        word2Ind[k] = idx
        Ind2word[idx] = k 
        idx += 1
    return word2Ind, Ind2word

In [12]:
word2Ind, Ind2word = get_dict(words)

word2Ind


{'am': 0, 'because': 1, 'happy': 2, 'i': 3, 'learning': 4}

In [13]:
Ind2word

{0: 'am', 1: 'because', 2: 'happy', 3: 'i', 4: 'learning'}

In [14]:
print("Index of the word 'i':", word2Ind['i'])

Index of the word 'i': 3


In [15]:
print("Word which has index 2: ", Ind2word[2])

Word which has index 2:  happy


In [16]:
V = len(word2Ind)
print("Size of vocabulary:", V)

Size of vocabulary: 5


In [17]:
n = word2Ind["happy"]
print(n)

2


In [18]:
import numpy as np
center_word_vector = np.zeros(V)
center_word_vector


array([0., 0., 0., 0., 0.])

In [19]:
len(center_word_vector) == V
V

5

In [20]:
center_word_vector[n] = 1
center_word_vector

array([0., 0., 1., 0., 0.])

In [21]:
def word_to_one_hot_vector(word, word2Ind, V):
    one_hot_vector = np.zeros(V)
    one_hot_vector[word2Ind[word]] = 1
    return one_hot_vector

word_to_one_hot_vector('happy', word2Ind, V)

array([0., 0., 1., 0., 0.])

In [22]:
word_to_one_hot_vector('learning', word2Ind, V)

array([0., 0., 0., 0., 1.])

In [23]:
context_words = ['i', 'am', 'because', 'i']
context_words_vectors = [word_to_one_hot_vector(w, word2Ind, V) for w in context_words]
context_words_vectors

[array([0., 0., 0., 1., 0.]),
 array([1., 0., 0., 0., 0.]),
 array([0., 1., 0., 0., 0.]),
 array([0., 0., 0., 1., 0.])]

In [24]:
np.mean(context_words_vectors, axis = 0)

array([0.25, 0.25, 0.  , 0.5 , 0.  ])

In [25]:
def context_words_to_vector(context_words, word2Ind, V):
    context_words_vectors = [word_to_one_hot_vector(w, word2Ind, V) for w in context_words]
    context_words_vectors = np.mean(context_words_vectors, axis = 0)
    return context_words_vectors

In [26]:
context_words_to_vector(['i', 'am', 'because', 'i'], word2Ind, V)
context_words_to_vector(['am', 'happy', 'i', 'am'], word2Ind, V)


array([0.5 , 0.  , 0.25, 0.25, 0.  ])

In [27]:
words

['i', 'am', 'happy', 'because', 'i', 'am', 'learning']

In [28]:
def get_training_example(words, C, word2Ind, V):
    for context_words, center_word in get_windows(words, C):
        yield context_words_to_vector(context_words, word2Ind, V), word_to_one_hot_vector(center_word, word2Ind, V)

In [29]:
for context_words_vector, center_word_vector in get_training_example(words, 2, word2Ind, V):
    print(f'Context words vector: {context_words_vector}')
    print(f'Center word vector: {center_word_vector}')
    print()

Context words vector: [0.25 0.25 0.   0.5  0.  ]
Center word vector: [0. 0. 1. 0. 0.]

Context words vector: [0.5  0.   0.25 0.25 0.  ]
Center word vector: [0. 1. 0. 0. 0.]

Context words vector: [0.25 0.25 0.25 0.   0.25]
Center word vector: [0. 0. 0. 1. 0.]



In [30]:
import numpy as np
np.random.seed(10)
z_1=10*np.random.rand(5,1)-5
z_1

array([[ 2.71320643],
       [-4.79248051],
       [ 1.33648235],
       [ 2.48803883],
       [-0.01492988]])

In [31]:
h=z_1.copy()
h<0
h[h<0]=0
h

array([[2.71320643],
       [0.        ],
       [1.33648235],
       [2.48803883],
       [0.        ]])

In [32]:
import numpy as np

def relu(z):
    result = z.copy()
    result[result < 0] = 0
    return result

# Example usage
np.random.seed(10)
z = 10 * np.random.rand(5, 1) - 5
print(relu(z))


[[2.71320643]
 [0.        ]
 [1.33648235]
 [2.48803883]
 [0.        ]]


In [33]:
def softmax(z):
    e_z=np.exp(z)
    sum_e_z=np.sum(e_z)
    return e_z/sum_e_z

In [34]:
softmax([9,8,11,10,8.5])

array([0.08276948, 0.03044919, 0.61158833, 0.22499077, 0.05020223])

In [35]:
np.sum(softmax([9,8,11,10,8.5]))==1

True

In [36]:
import numpy as np
from scipy import linalg
from collections import defaultdict


def sigmoid(z):
    # sigmoid function
    return 1.0 / (1.0 + np.exp(-z))


def get_idx(words, word2Ind):
    idx = []
    for word in words:
        idx = idx + [word2Ind[word]]
    return idx


def pack_idx_with_frequency(context_words, word2Ind):
    freq_dict = defaultdict(int)
    for word in context_words:
        freq_dict[word] += 1
    idxs = get_idx(context_words, word2Ind)
    packed = []
    for i in range(len(idxs)):
        idx = idxs[i]
        freq = freq_dict[context_words[i]]
        packed.append((idx, freq))
    return packed


def get_vectors(data, word2Ind, V, C):
    i = C
    while True:
        y = np.zeros(V)
        x = np.zeros(V)
        center_word = data[i]
        y[word2Ind[center_word]] = 1
        context_words = data[(i - C) : i] + data[(i + 1) : (i + C + 1)]
        num_ctx_words = len(context_words)
        for idx, freq in pack_idx_with_frequency(context_words, word2Ind):
            x[idx] = freq / num_ctx_words
        yield x, y
        i += 1
        if i >= len(data):
            print("i is being set to 0")
            i = 0


def get_batches(data, word2Ind, V, C, batch_size):
    batch_x = []
    batch_y = []
    for x, y in get_vectors(data, word2Ind, V, C):
        while len(batch_x) < batch_size:
            batch_x.append(x)
            batch_y.append(y)
        else:
            yield np.array(batch_x).T, np.array(batch_y).T
            batch_x = []
            batch_y = []


def compute_pca(data, n_components=2):
    """
    Input: 
        data: of dimension (m,n) where each row corresponds to a word vector
        n_components: Number of components you want to keep.
    Output: 
        X_reduced: data transformed in 2 dims/columns + regenerated original data
    pass in: data as 2D NumPy array
    """

    m, n = data.shape

    ### START CODE HERE ###
    # mean center the data
    data -= data.mean(axis=0)
    # calculate the covariance matrix
    R = np.cov(data, rowvar=False)
    # calculate eigenvectors & eigenvalues of the covariance matrix
    # use 'eigh' rather than 'eig' since R is symmetric,
    # the performance gain is substantial
    evals, evecs = linalg.eigh(R)
    # sort eigenvalue in decreasing order
    # this returns the corresponding indices of evals and evecs
    idx = np.argsort(evals)[::-1]

    evecs = evecs[:, idx]
    # sort eigenvectors according to same index
    evals = evals[idx]
    # select the first n eigenvectors (n is desired dimension
    # of rescaled data array, or dims_rescaled_data)
    evecs = evecs[:, :n_components]
    ### END CODE HERE ###
    return np.dot(evecs.T, data.T).T


def get_dict(data):
    """
    Input:
        K: the number of negative samples
        data: the data you want to pull from
        indices: a list of word indices
    Output:
        word_dict: a dictionary with the weighted probabilities of each word
        word2Ind: returns dictionary mapping the word to its index
        Ind2Word: returns dictionary mapping the index to its word
    """
    #
    #     words = nltk.word_tokenize(data)
    words = sorted(list(set(data)))
    n = len(words)
    idx = 0
    # return these correctly
    word2Ind = {}
    Ind2word = {}
    for k in words:
        word2Ind[k] = idx
        Ind2word[idx] = k
        idx += 1
    return word2Ind, Ind2word

In [37]:
import numpy as np
n=3
v=5


In [38]:
# Define first matrix of weights
W1 = np.array([[ 0.41687358,  0.08854191, -0.23495225,  0.28320538,  0.41800106],
               [ 0.32735501,  0.22795148, -0.23951958,  0.4117634 , -0.23924344],
               [ 0.26637602, -0.23846886, -0.37770863, -0.11399446,  0.34008124]])

# Define second matrix of weights
W2 = np.array([[-0.22182064, -0.43008631,  0.13310965],
               [ 0.08476603,  0.08123194,  0.1772054 ],
               [ 0.1871551 , -0.06107263, -0.1790735 ],
               [ 0.07055222, -0.02015138,  0.36107434],
               [ 0.33480474, -0.39423389, -0.43959196]])

# Define first vector of biases
b1 = np.array([[ 0.09688219],
               [ 0.29239497],
               [-0.27364426]])

# Define second vector of biases
b2 = np.array([[ 0.0352008 ],
               [-0.36393384],
               [-0.12775555],
               [-0.34802326],
               [-0.07017815]])

In [39]:
print(v)
print(n)
print(W1.shape)
print(b1.shape)
print(W2.shape)
print(b2.shape)

5
3
(3, 5)
(3, 1)
(5, 3)
(5, 1)


In [40]:
word=['i' ,'am' ,'happy', 'because', 'i' ,'am' ,'learning']
word2Ind,Ind2word=get_dict(words)

print(word2Ind)
print(Ind2word)

{'am': 0, 'because': 1, 'happy': 2, 'i': 3, 'learning': 4}
{0: 'am', 1: 'because', 2: 'happy', 3: 'i', 4: 'learning'}


In [41]:
def get_windows(words, C):
    i = C
    while i < len(words) - C:
        center_word = words[i]
        context_words = words[(i-C):i] + words[(i+1):(i + C + 1)]
        yield context_words, center_word
        i +=1 

In [42]:
def word_to_one_hot_vector(word, word2Ind, V):
    one_hot_vector = np.zeros(V)
    one_hot_vector[word2Ind[word]] = 1
    return one_hot_vector


In [43]:
def context_words_to_vector(context_words, word2Ind, V):
    context_words_vectors = [word_to_one_hot_vector(w, word2Ind, V) for w in context_words]
    context_words_vectors = np.mean(context_words_vectors, axis = 0)
    return context_words_vectors

In [44]:
def get_training_example(words, C, word2Ind, V):
    for context_words, center_word in get_windows(words, C):
        yield context_words_to_vector(context_words, word2Ind, V), word_to_one_hot_vector(center_word, word2Ind, V)

In [45]:
training_examples=get_training_example(words,2,word2Ind,v)
training_examples

<generator object get_training_example at 0x0000018A7EB849D0>

In [46]:
x_array,y_array=next(training_examples)
print(x_array)

[0.25 0.25 0.   0.5  0.  ]


In [47]:
print(y_array)

[0. 0. 1. 0. 0.]


In [48]:
x=x_array.copy()
x.shape=(v,1)
print(x)
y=y_array.copy()
y.shape=(v,1)
print(y)

[[0.25]
 [0.25]
 [0.  ]
 [0.5 ]
 [0.  ]]
[[0.]
 [0.]
 [1.]
 [0.]
 [0.]]


In [49]:
import numpy as np

def relu(z):
    result = z.copy()
    result[result < 0] = 0
    return result

# Example usage
np.random.seed(10)
z = 10 * np.random.rand(5, 1) - 5
print(relu(z))

def softmax(z):
    e_z=np.exp(z)
    sum_e_z=np.sum(e_z)
    return e_z/sum_e_z

[[2.71320643]
 [0.        ]
 [1.33648235]
 [2.48803883]
 [0.        ]]


In [50]:
z1=np.dot(W1,x)+b1
print(z1)

[[ 0.36483875]
 [ 0.63710329]
 [-0.3236647 ]]


In [51]:
h=relu(z1)
h

array([[0.36483875],
       [0.63710329],
       [0.        ]])

In [52]:
z2=np.dot(W2,h)+b2
z2

array([[-0.31973737],
       [-0.28125477],
       [-0.09838369],
       [-0.33512159],
       [-0.19919612]])

In [53]:
y_hat=softmax(z2)
y_hat

array([[0.18519074],
       [0.19245626],
       [0.23107446],
       [0.18236353],
       [0.20891502]])

In [54]:
y

array([[0.],
       [0.],
       [1.],
       [0.],
       [0.]])

In [55]:
def cross_entropy_loss(y_predicted,y_actual):
    loss=None
    return loss

In [56]:
def compute_cost(y,y_hat,batch_size):
    logprobs=np.multiply(np.log(y_hat),y)+np.multiply(np.log(1-y_hat),1-y)
    cost=-1/batch_size*np.sum(logprobs)
    cost=np.squeeze(cost)
    return cost

In [57]:
compute_cost(y,y_hat,1)

2.3192619086254354

In [58]:
grad_b2=y_hat-y
grad_b2

array([[ 0.18519074],
       [ 0.19245626],
       [-0.76892554],
       [ 0.18236353],
       [ 0.20891502]])

In [59]:
grad_W2=np.dot(y_hat-y,h.T)
grad_W2

array([[ 0.06756476,  0.11798563,  0.        ],
       [ 0.0702155 ,  0.12261452,  0.        ],
       [-0.28053384, -0.48988499,  0.        ],
       [ 0.06653328,  0.1161844 ,  0.        ],
       [ 0.07622029,  0.13310045,  0.        ]])

In [60]:
grad_b1=relu(np.dot(W2.T,y_hat-y))
grad_b1

array([[0.        ],
       [0.        ],
       [0.17045858]])

In [61]:
grad_W1=np.dot(relu(np.dot(W2.T,y_hat-y)),x.T)
grad_W1

array([[0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.04261464, 0.04261464, 0.        , 0.08522929, 0.        ]])

In [62]:
print(v)
print(n)
print(grad_W1.shape)
print(grad_b1.shape)
print(grad_W2.shape)
print(grad_b2.shape)

5
3
(3, 5)
(3, 1)
(5, 3)
(5, 1)


In [63]:
alpha=0.03
W1_new=W1-alpha*grad_W1

In [64]:
print(W1)
print(W1_new)

[[ 0.41687358  0.08854191 -0.23495225  0.28320538  0.41800106]
 [ 0.32735501  0.22795148 -0.23951958  0.4117634  -0.23924344]
 [ 0.26637602 -0.23846886 -0.37770863 -0.11399446  0.34008124]]
[[ 0.41687358  0.08854191 -0.23495225  0.28320538  0.41800106]
 [ 0.32735501  0.22795148 -0.23951958  0.4117634  -0.23924344]
 [ 0.26509758 -0.2397473  -0.37770863 -0.11655134  0.34008124]]


In [71]:
import nltk 
from nltk.tokenize import word_tokenize
import numpy as np
from collections import Counter
# from utils2 import sigmoid, get_batches, compute_pca, get_dict

In [72]:
nltk.data.path.append(".")

In [73]:
import re
with open('shakespeare.txt') as f:
    data = f.read()
data=re.sub(r'[,!?;-]','.',data)
data=nltk.word_tokenize(data)
data=[ch.lower() for ch in data if ch.isalpha() or ch =='.']
print('number of tokens:',len(data),'\n',data[:15])

number of tokens: 60976 
 ['ï', 'for', 'a', 'muse', 'of', 'fire', '.', 'that', 'would', 'ascend', 'the', 'brightest', 'heaven', 'of', 'invention']


In [74]:
fdist=nltk.FreqDist(word for word in data)
print(len(fdist))
print(fdist.most_common(20))

5776
[('.', 9630), ('the', 1521), ('and', 1394), ('i', 1257), ('to', 1159), ('of', 1093), ('my', 857), ('that', 781), ('in', 770), ('a', 752), ('you', 748), ('is', 630), ('not', 559), ('for', 467), ('it', 460), ('with', 441), ('his', 434), ('but', 417), ('me', 417), ('your', 397)]


In [80]:
word2Ind,Ind2word=get_dist(data)
V=len(word2Ind)
print(V)

NameError: name 'get_dist' is not defined

In [81]:
print(word2Ind['king'])
print(Ind2word([2743]))

2744


TypeError: 'dict' object is not callable

In [82]:
def initialize_model(N,V,random_seed=1):
    np.random.seed(random_seed)
    
    W1=np.random.rand(N,V)
    W2=np.random.rand(V,N)
    b1=np.random.rand(N,1)
    b2=np.ramdom.rand(V,1)
    
    return W1,W2,b1,b2
    

In [79]:
tmp_N=4
tmp_V=10
tmp_W1,tmp_W2,tmp_b1,tmp_b2=initialize_model(tmp_N,tmp_V)
assert tmp_W1.shape==((tmp_N,tmp_V))
assert tmp_W2.shape==((tmp_V,tmp_N))
print(tmp_W1.shape)
print(tmp_W2.shape)
print(tmp_b1.shape)
print(tmp_W1.shape)

AttributeError: module 'numpy' has no attribute 'ramdom'

In [ ]:
def softmax(z):
    e_z=np.exp(z)
    yhat=z/np.sum(e_z.axis=0)
    return yhat

In [ ]:
tmp=np.array([[1,2,3],[1,1,1]])
tmp_sm=softmax(tmp)
display(tmp_sm)


In [83]:
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
from collections import Counter

# Set up NLTK data path to avoid download issues
nltk.data.path.append(".")

import re

# Reading and preprocessing the data
with open('shakespeare.txt') as f:
    data = f.read()

# Replacing punctuation marks with a period and tokenizing the data
data = re.sub(r'[,!?;-]', '.', data)
data = nltk.word_tokenize(data)

# Convert tokens to lowercase, and keep only alphabetic characters or periods
data = [ch.lower() for ch in data if ch.isalpha() or ch == '.']

# Print the number of tokens and a sample of the data
print('Number of tokens:', len(data), '\n', data[:15])

# Generate a frequency distribution of the tokens
fdist = nltk.FreqDist(word for word in data)
print(len(fdist))
print(fdist.most_common(20))

# Function to generate word-to-index and index-to-word dictionaries
def get_dict(data):
    vocab = set(data)
    word2Ind = {word: idx for idx, word in enumerate(vocab)}
    Ind2word = {idx: word for word, idx in word2Ind.items()}
    return word2Ind, Ind2word

# Generate the word-index dictionaries
word2Ind, Ind2word = get_dict(data)
V = len(word2Ind)
print(V)
print(word2Ind['king'])
print(Ind2word[2743])

# Function to initialize the model with random weights
def initialize_model(N, V, random_seed=1):
    np.random.seed(random_seed)
    
    W1 = np.random.rand(N, V)
    W2 = np.random.rand(V, N)
    b1 = np.random.rand(N, 1)
    b2 = np.random.rand(V, 1)  # Corrected typo: changed 'ramdom' to 'random'
    
    return W1, W2, b1, b2

# Testing the initialization function
tmp_N = 4
tmp_V = 10
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N, tmp_V)

# Asserting the shapes of the initialized matrices and biases
assert tmp_W1.shape == (tmp_N, tmp_V)
assert tmp_W2.shape == (tmp_V, tmp_N)
assert tmp_b1.shape == (tmp_N, 1)
assert tmp_b2.shape == (tmp_V, 1)

# Print the shapes
print(tmp_W1.shape)
print(tmp_W2.shape)
print(tmp_b1.shape)
print(tmp_b2.shape)  # Changed print statement to print the correct variable


Number of tokens: 60976 
 ['ï', 'for', 'a', 'muse', 'of', 'fire', '.', 'that', 'would', 'ascend', 'the', 'brightest', 'heaven', 'of', 'invention']
5776
[('.', 9630), ('the', 1521), ('and', 1394), ('i', 1257), ('to', 1159), ('of', 1093), ('my', 857), ('that', 781), ('in', 770), ('a', 752), ('you', 748), ('is', 630), ('not', 559), ('for', 467), ('it', 460), ('with', 441), ('his', 434), ('but', 417), ('me', 417), ('your', 397)]
5776
2058
drive
(4, 10)
(10, 4)
(4, 1)
(10, 1)


In [86]:
import numpy as np

# Softmax function
def softmax(z):
    e_z = np.exp(z - np.max(z, axis=0))  # Subtracting max for numerical stability
    yhat = e_z / np.sum(e_z, axis=0)
    return yhat

# Test with an example array
tmp = np.array([[1, 2, 3], [1, 1, 1]])
tmp_sm = softmax(tmp)

# Display the result
print(tmp_sm)


[[0.5        0.73105858 0.88079708]
 [0.5        0.26894142 0.11920292]]


In [90]:
import numpy as np

# Forward propagation function
def forward_prop(x, W1, W2, b1, b2):
    # Linear combination for the hidden layer
    h = np.dot(W1, x) + b1
    
    # Apply ReLU activation
    h = np.maximum(0, h)
    
    # Linear combination for the output layer
    z = np.dot(W2, h) + b2
    return z, h

# Initialize the model parameters
tmp_N = 2
tmp_V = 3
tmp_x = np.array([[0, 1, 0]]).T

# Initialize the weights and biases
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(N=tmp_N, V=tmp_V, random_seed=1)

# Print shapes for verification
print("Shape of input x:", tmp_x.shape)
print("Number of neurons in hidden layer (N):", tmp_N)

# Perform forward propagation
tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)

# Print the output and hidden layer activations
print("\nShape of output z:", tmp_z.shape)
print("Output z:", tmp_z)

print("\nShape of hidden activations h:", tmp_h.shape)
print("Hidden activations h:", tmp_h)


Shape of input x: (3, 1)
Number of neurons in hidden layer (N): 2

Shape of output z: (3, 1)
Output z: [[0.55379268]
 [1.58960774]
 [1.50722933]]

Shape of hidden activations h: (2, 1)
Hidden activations h: [[0.92477674]
 [1.02487333]]


In [91]:
def compute_cost(y,yhat,batch_size):
    logprobs=np.multiply(np.log(yhat),y)+np.multiply(np.log(1-yhat),1-y)
    cost=-1/batch_size*np.sum(logprobs)
    cost=np.squeeze(cost)
    return cost

In [97]:
tmp_C=2
tmp_N=50
tmp_batch_size=4
tmp_word2Ind,tmp_Ind2word=get_dict(data)
tmp_V=len(word2Ind)

tmp_x,tmp_y=next(get_batches(data,tmp_word2Ind,tmp_V,tmp_C,tmp_batch_size))

print(tmp_x.shape)
print(tmp_y.shape)

tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N,tmp_V)

print(tmp_W1.shape)
print(tmp_W2.shape)
print(tmp_b1.shape)
print(tmp_b2.shape)  

tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)
print(tmp_z.shape)
print(tmp_h.shape)

tmp_yhat=softmax(tmp_z)
print(tmp_yhat.shape)

tmp_cost=compute_cost(tmp_y,tmp_yhat,tmp_batch_size)
print(tmp_cost)

(5776, 4)
(5776, 4)
(50, 5776)
(5776, 50)
(50, 1)
(5776, 1)
(5776, 4)
(50, 4)
(5776, 4)
13.008015863077702


In [98]:
import numpy as np

# Function to generate word-to-index and index-to-word dictionaries
def get_dict(data):
    vocab = set(data)
    word2Ind = {word: idx for idx, word in enumerate(vocab)}
    Ind2word = {idx: word for word, idx in word2Ind.items()}
    return word2Ind, Ind2word

# Batch generator function
def get_batches(data, word2Ind, V, C, batch_size):
    num_batches = len(data) // batch_size
    for i in range(0, num_batches * batch_size, batch_size):
        x_batch = np.zeros((V, batch_size))
        y_batch = np.zeros((V, batch_size))
        
        for j in range(batch_size):
            word = data[i + j]
            word_idx = word2Ind[word]
            x_batch[word_idx, j] = 1  # One-hot encoding for input
            # Here you can set up the target/output 'y' based on your task
            y_batch[word_idx, j] = 1  # Placeholder, adjust as needed

        yield x_batch, y_batch

# Initialize the model parameters
def initialize_model(N, V, random_seed=1):
    np.random.seed(random_seed)
    W1 = np.random.rand(N, V)
    W2 = np.random.rand(V, N)
    b1 = np.random.rand(N, 1)
    b2 = np.random.rand(V, 1)
    return W1, W2, b1, b2

# Forward propagation function
def forward_prop(x, W1, W2, b1, b2):
    h = np.dot(W1, x) + b1
    h = np.maximum(0, h)  # ReLU activation
    z = np.dot(W2, h) + b2
    return z, h

# Softmax function
def softmax(z):
    e_z = np.exp(z - np.max(z, axis=0))  # Numerical stability
    yhat = e_z / np.sum(e_z, axis=0)
    return yhat

# Compute cost function (binary cross-entropy loss)
def compute_cost(y, yhat, batch_size):
    logprobs = np.multiply(np.log(yhat), y) + np.multiply(np.log(1 - yhat), 1 - y)
    cost = -1 / batch_size * np.sum(logprobs)
    cost = np.squeeze(cost)
    return cost

# Example usage:

# Assume 'data' has already been loaded and preprocessed
tmp_C = 2
tmp_N = 50
tmp_batch_size = 4
tmp_word2Ind, tmp_Ind2word = get_dict(data)
tmp_V = len(tmp_word2Ind)

# Get a batch of input and output
tmp_x, tmp_y = next(get_batches(data, tmp_word2Ind, tmp_V, tmp_C, tmp_batch_size))

print("Shape of tmp_x:", tmp_x.shape)
print("Shape of tmp_y:", tmp_y.shape)

# Initialize weights and biases
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N, tmp_V)

print("Shape of W1:", tmp_W1.shape)
print("Shape of W2:", tmp_W2.shape)
print("Shape of b1:", tmp_b1.shape)
print("Shape of b2:", tmp_b2.shape)

# Forward propagation
tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)
print("Shape of z:", tmp_z.shape)
print("Shape of h:", tmp_h.shape)

# Softmax output
tmp_yhat = softmax(tmp_z)
print("Shape of yhat:", tmp_yhat.shape)

# Compute the cost
tmp_cost = compute_cost(tmp_y, tmp_yhat, tmp_batch_size)
print("Cost:", tmp_cost)


Shape of tmp_x: (5776, 4)
Shape of tmp_y: (5776, 4)
Shape of W1: (50, 5776)
Shape of W2: (5776, 50)
Shape of b1: (50, 1)
Shape of b2: (5776, 1)
Shape of z: (5776, 4)
Shape of h: (50, 4)
Shape of yhat: (5776, 4)
Cost: 13.008015863077702


In [100]:
# Backpropagation function to compute gradients
def back_probs(x, yhat, y, h, W1, W2, batch_size):
    # Compute gradient for W2
    grad_W2 = (1 / batch_size) * np.dot((yhat - y), h.T)
    
    # Compute gradient for b2
    grad_b2 = (1 / batch_size) * np.sum(yhat - y, axis=1, keepdims=True)
    
    # Compute gradient for hidden layer (before activation)
    l1 = np.dot(W2.T, (yhat - y))
    
    # Apply ReLU derivative
    l1 = l1 * (h > 0)  # Derivative of ReLU: 1 where h > 0, else 0
    
    # Compute gradient for W1
    grad_W1 = (1 / batch_size) * np.dot(l1, x.T)
    
    # Compute gradient for b1
    grad_b1 = (1 / batch_size) * np.sum(l1, axis=1, keepdims=True)
    
    return grad_W1, grad_W2, grad_b1, grad_b2


In [101]:
# Assume 'data' has already been loaded and preprocessed
tmp_C = 2
tmp_N = 50
tmp_batch_size = 4
tmp_word2Ind, tmp_Ind2word = get_dict(data)
tmp_V = len(tmp_word2Ind)

# Get a batch of input and output
tmp_x, tmp_y = next(get_batches(data, tmp_word2Ind, tmp_V, tmp_C, tmp_batch_size))

# Initialize weights and biases
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N, tmp_V)

# Forward propagation
tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)

# Softmax output
tmp_yhat = softmax(tmp_z)

# Backpropagation to compute gradients
grad_W1, grad_W2, grad_b1, grad_b2 = back_probs(tmp_x, tmp_yhat, tmp_y, tmp_h, tmp_W1, tmp_W2, tmp_batch_size)

# Print shapes of gradients to verify correctness
print("Shape of grad_W1:", grad_W1.shape)
print("Shape of grad_W2:", grad_W2.shape)
print("Shape of grad_b1:", grad_b1.shape)
print("Shape of grad_b2:", grad_b2.shape)


Shape of grad_W1: (50, 5776)
Shape of grad_W2: (5776, 50)
Shape of grad_b1: (50, 1)
Shape of grad_b2: (5776, 1)


In [102]:
# Forward propagation
print()
tmp_z, tmp_h = forword_prob(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)
print("Shape of tmp_z:", tmp_z.shape)
print("Shape of tmp_h:", tmp_h.shape)

# Apply softmax to compute yhat (predictions)
print()
tmp_yhat = softmax(tmp_z)
print("Shape of tmp_yhat:", tmp_yhat.shape)

# Define the effective number of classes (in this example, it’s 2 * tmp_C)
tmp_m = (2 * tmp_C)

# Backpropagation: compute gradients
tmp_grad_W1, tmp_grad_W2, tmp_grad_b1, tmp_grad_b2 = back_probs(
    tmp_x,  # Input data
    tmp_yhat,  # Predictions from softmax
    tmp_y,  # True labels
    tmp_h,  # Hidden layer activations
    tmp_W1,  # Weight matrix W1
    tmp_W2,  # Weight matrix W2
    tmp_batch_size  # Batch size for the gradient computation
)

# Print shapes of computed gradients to verify correctness
print("Shape of grad_W1:", tmp_grad_W1.shape)
print("Shape of grad_W2:", tmp_grad_W2.shape)
print("Shape of grad_b1:", tmp_grad_b1.shape)
print("Shape of grad_b2:", tmp_grad_b2.shape)



Shape of tmp_z: (5776, 4)
Shape of tmp_h: (50, 4)

Shape of tmp_yhat: (5776, 4)
Shape of grad_W1: (50, 5776)
Shape of grad_W2: (5776, 50)
Shape of grad_b1: (50, 1)
Shape of grad_b2: (5776, 1)


In [103]:
def gradient_decent(data, word2Ind, N, V, num_iters, alpha=0.03):
    """
    Perform gradient descent to optimize model parameters.

    Args:
    data: Input text data.
    word2Ind: Dictionary mapping words to indices.
    N: Size of the hidden layer (number of neurons).
    V: Vocabulary size (number of unique words).
    num_iters: Number of iterations for training.
    alpha: Learning rate (default: 0.03).

    Returns:
    Trained weights and biases (W1, W2, b1, b2).
    """
    # Initialize weights and biases with a fixed random seed
    W1, W2, b1, b2 = initialize_model(N, V, random_seed=20)
    
    # Define batch size and initial iteration count
    batch_size = 128
    C = 2  # Number of classes (binary classification in this case)
    iters = 0  # Iteration counter

    # Training loop for num_iters iterations
    while iters < num_iters:
        # Iterate through batches of data
        for x, y in get_batches(data, word2Ind, V, C, batch_size):
            # Forward propagation: compute predictions (yhat)
            z, h = forword_prob(x, W1, W2, b1, b2)
            yhat = softmax(z)

            # Compute the cost (loss)
            cost = compute_cost(y, yhat, batch_size)

            # Print cost every 10 iterations
            if (iters + 1) % 10 == 0:
                print(f"Iteration {iters + 1}: Cost = {cost}")

            # Backpropagation: compute gradients for W1, W2, b1, and b2
            grad_W1, grad_W2, grad_b1, grad_b2 = back_probs(x, yhat, y, h, W1, W2, batch_size)

            # Gradient descent: update weights and biases
            W1 -= alpha * grad_W1
            W2 -= alpha * grad_W2
            b1 -= alpha * grad_b1
            b2 -= alpha * grad_b2

            # Increment iteration counter
            iters += 1

            # Break if we reach the maximum number of iterations
            if iters >= num_iters:
                break

        # Adjust learning rate every 100 iterations
        if iters % 100 == 0:
            alpha *= 0.66

    # Return the trained weights and biases
    return W1, W2, b1, b2
